# cv-data-lesson T4 GPU実モデル受け入れテスト

授業用Notebookとは別の確認用Notebookです。T4 GPUランタイムで、固定コミットのライブラリを導入し、ResNet分類・数値予測、DETR、SAMを実際に実行します。最後のSDXLセルは大きなモデルを取得するため、必要なときだけ明示的に実行します。モデルキャッシュはColabの`/content`に置き、リポジトリへ保存しません。

In [ ]:
LIBRARY_REF = "d7a45857394a72f83ab599df9465a8390cfff10f"
%pip install -q "cv-data-lesson[torch,vision,diffusion] @ git+https://github.com/708san/CV_data.git@2977753b48e1d1ab97797dcbb015d61fbdf592cb"

In [ ]:
import importlib.metadata as metadata
from pathlib import Path
import torch
from PIL import Image
from cvlesson import generation, models, vision

assert torch.cuda.is_available(), "T4 GPUランタイムを選択してください"
assert torch.cuda.device_count() >= 1
print("version:", metadata.version("cv-data-lesson"))
print("GPU:", torch.cuda.get_device_name(0))
print("ResNet devices:", models.available_devices())
print("Vision devices:", vision.available_devices())
assert models.available_devices()["cuda"] is True
assert vision.available_devices()["cuda"] is True
MODEL_CACHE = Path("/content/cvlesson-gpu-model-cache")
MODEL_CACHE.mkdir(parents=True, exist_ok=True)

## ResNet分類・数値予測

ImageNet重みは取得せず、`weights=None`で小さな合成画像を使います。GPUへの移動、学習、予測、エポック進捗表示、分類・回帰の両方を確認します。

In [ ]:
synthetic_images = [
    Image.new("RGB", (64, 64), color)
    for color in ("red", "red", "blue", "blue")
]
classification_model = models.ResNetClassifier(
    device="cuda", weights=None, epochs=1, batch_size=2, verbose=True, random_state=1
)
classification_model.fit(synthetic_images, [0, 0, 1, 1])
classification_predictions = classification_model.predict(synthetic_images)
assert classification_model.device == "cuda"
assert len(classification_predictions) == 4
assert classification_model.loss_history_ and len(classification_model.loss_history_) == 1
print("ResNet classification on T4: OK", classification_predictions.tolist())

In [ ]:
regression_model = models.ResNetRegressor(
    device="cuda", weights=None, epochs=1, batch_size=2, verbose=True, random_state=1
)
regression_model.fit(synthetic_images, [0.1, 0.2, 0.8, 0.9])
regression_predictions = regression_model.predict(synthetic_images)
assert regression_model.device == "cuda"
assert len(regression_predictions) == 4
assert regression_model.loss_history_ and len(regression_model.loss_history_) == 1
print("ResNet regression on T4: OK", regression_predictions.tolist())

## DETR物体検出

固定revisionのDETRをT4へ読み込み、画像座標の検出結果を返すことを確認します。モデルの一般性能や精度評価ではありません。

In [ ]:
detector = vision.load_detector(device="cuda", cache_dir=MODEL_CACHE)
detection_image = Image.new("RGB", (640, 480), "white")
detections = detector.predict(detection_image, threshold=0.9)
assert detector.device == "cuda"
assert isinstance(detections, list)
for detection in detections:
    assert detection.label and 0 <= detection.score <= 1
    assert len(detection.box) == 4
print("DETR on T4: OK; detections:", len(detections))

## SAM領域分割

固定revisionのSAMをT4へ読み込み、矩形promptから画像サイズと一致するマスクを返すことを確認します。

In [ ]:
segmenter = vision.load_segmenter(device="cuda", cache_dir=MODEL_CACHE)
segmentation_image = Image.new("RGB", (640, 480), "white")
segmentation = segmenter.predict(segmentation_image, box=(100, 100, 500, 380))
assert segmenter.device == "cuda"
assert segmentation.mask.shape == (480, 640)
assert segmentation.prompt_type == "box"
print("SAM on T4: OK; mask score:", segmentation.score)

## SDXL画像生成（必要なときだけ）

このセルはSDXL基盤モデルを取得するため、実行前に時間・ディスク・GPUメモリを確認してください。`RUN_SDXL = True`へ変更した場合だけ実行します。

In [ ]:
RUN_SDXL = False
if RUN_SDXL:
    generator = generation.load(
        device="cuda", lora=None, cache_dir=MODEL_CACHE
    )
    generated_image = generator.generate(
        "a simple watercolor illustration of an orange on a table",
        num_inference_steps=2, width=512, height=512, seed=7,
    )
    assert generator.device == "cuda"
    assert generated_image.size == (512, 512)
    print("SDXL on T4: OK", generated_image.size)
else:
    print("SDXLは未実行。必要なときだけRUN_SDXL=Trueに変更してください")

## 判定

ResNet・DETR・SAMのセルが成功すれば、T4上の実モデルでdevice別名、GPU移動、学習・推論の基本経路を確認できています。SDXLは`RUN_SDXL=True`で実行した場合のみ、実際の画像生成まで確認できます。